# Exercises

## Exercise 14-2
Write a function called sed that takes as arguments a pattern string, a replacement
string, and two filenames; it should read the first file and write the contents into the
second file (creating it if necessary). If the pattern string appears anywhere in the file,
it should be replaced with the replacement string.
If an error occurs while opening, reading, writing, or closing files, your program
should catch the exception, print an error message, and exit.

In [2]:
using Revise

In [3]:
function sed(pat_string, rep_string, file_a; file_b::String = "Default_file.txt")
    try
    open(file_a, "r") do file1
        content_a = read(file1, String)

        updated_content = replace(content_a, pat_string => rep_string)

        open(file_b, "w") do file2
            write(file2, updated_content)
        end
    end

    catch
        println("Something went wrong")
    end
    println("Pattern successfully replaced to file $file_b")
end

sed (generic function with 1 method)

In [4]:
pat_string = "\r"
rep_string = "\n"
file_a = "output.txt"

sed(pat_string, rep_string, file_a)

Pattern successfully replaced to file Default_file.txt


In [5]:
function anagram(file)
    list = Dict()
    for word in eachline(file)
        list[word] = []
    end
    
    for word_now in keys(list) 
        for a in keys(list)
            if sort(collect(word_now)) == sort(collect(a))
                push!(list[word_now], a) 
            end
        end
    end

    list
end

println(anagram("Test.txt"))

Dict{Any, Any}("conserve" => Any["conserve", "converse"], "aasvogels" => Any["aasvogels"], "aahing" => Any["aahing"], "aah" => Any["aah"], "csuaba" => Any["csuaba", "abacus"], "baa" => Any["baa", "aba"], "abamperes" => Any["abamperes"], "abampere" => Any["abampere"], "abalone" => Any["abalone"], "aaliis" => Any["aaliis"], "abaft" => Any["abaft"], "aal" => Any["aal"], "aardvark" => Any["aardvark"], "aahs" => Any["aahs"], "aals" => Any["aals", "slaa"], "aa" => Any["aa"], "abakas" => Any["abakas"], "aasvogel" => Any["aasvogel"], "abamps" => Any["abamps"], "abaca" => Any["abaca"], "abamp" => Any["abamp", "pamba"], "abaci" => Any["abaci", "aciba"], "abacuses" => Any["abacuses"], "aas" => Any["aas"], "aahed" => Any["aahed"], "abacas" => Any["abacas"], "pamba" => Any["abamp", "pamba"], "aba" => Any["baa", "aba"], "aardwolf" => Any["aardwolf"], "aback" => Any["aback"], "aciba" => Any["abaci", "aciba"], "abaka" => Any["abaka"], "abalones" => Any["abalones"], "aband" => Any["aband"], "aalii" => 

## Exercise 14-3
If you have done “Exercise 12-3” on page 150, you’ll see that a dictionary is created
that maps from a sorted string of letters to the array of words that can be spelled with
those letters. For example, "opst" maps to the array ["opts", "post", "pots",
"spot", "stop", "tops"].
Write a module that imports anagramsets and provides two new functions:
storeanagrams should store the anagram dictionary using JLD2, and readanagrams
should look up a word and return an array of its anagrams.

In [10]:
anarg = anagram("Test.txt")


module anagramsets
    using JLD2, FileIO

    function storeanargrams(filename)
        # @show Main.anarg
        save(filename, Main.anarg) 
        return filename
    end
    
    function readanargrams(filename, look_word)
        data = storeanargrams(filename)
        return load(data, look_word)
    end
end

using .anagramsets

try
    read_data = anagramsets.readanargrams("Anarg_data.jld2", "conserve")
    println(read_data)
catch
    @warn "Word not found"
end


Any["conserve", "converse"]


## Exercise 14-4
In a large collection of MP3 files, there may be more than one copy of the same song,
stored in different directories or with different filenames. The goal of this exercise is
to search for duplicates.
1. Write a program that searches a directory and all of its subdirectories, recursively,
and returns an array of complete paths for all files with a given suffix
(like .mp3).
2. To recognize duplicates, you can use md5sum or md5 to compute a “checksum” for
each file. If two files have the same checksum, they probably have the same contents.
3. To double-check, you can use the Unix command diff.

1. Write a program that searches a directory and all of its subdirectories, recursively,
and returns an array of complete paths for all files with a given suffix
(like .mp3).

In [ ]:
function checkdir(dir, arr)
    dir_content = readdir(dir, join = true)
    for i in dir_content
        if isfile(i)
            push!(arr, basename(i))
        else
            checkdir(i, arr)
        end
    end
    return arr
end

checkdir (generic function with 2 methods)

In [32]:
dir = "c:\\Users\\goodn\\Desktop\\Julia-Learn\\music"

arr = []
result = checkdir(dir, arr)
println(result)

Any["Hillsong_Worship_-_This_is_The_Air_I_Breathe_CeeNaija.com_.mp3", "Hillsong_Worship_-_Unending_Love_CeeNaija.com_.mp3", "Jenn_Johnson_Bethel_-_Tremble_CeeNaija.com_.mp3", "Hillsong_Worship_-_Hope_Of_The_Ages_Ft_Jad_Gillies_CeeNaija.com_.mp3", "Hillsong_Worship_-_Love_On_The_Line_CeeNaija.com_.mp3", "Bethel_Music_-_Fall_Like_Rain_Ft_Zahriya_Zachary_CeeNaija.com_.mp3", "Brian_Jenn_Johnson_Bethel_-_After_All_These_Years_CeeNaija.com_.mp3", "Brian_Jenn_Johnson_Bethel_-_Youre_Gonna_Be_Okay_CeeNaija.com_.mp3"]


2. To recognize duplicates, you can use md5sum or md5 to compute a “checksum” for
each file. If two files have the same checksum, they probably have the same contents.
3. To double-check, you can use the Unix command diff.

In [30]:
using SHA

function checkdir(dir, arr)
    dir_content = readdir(dir, join = true)
    for i in dir_content
        if !(any(a -> a[2] == sha256(i), arr) && isfile(i))
            push!(arr, (basename(i), sha256(i)))
        else
            checkdir(i, arr)
        end
    end
    result = map(t -> t[1], arr)
    return result
end

checkdir (generic function with 1 method)

In [31]:
dir = "c:\\Users\\goodn\\Desktop\\Julia-Learn\\music"

arr = []
result = checkdir(dir, arr)
println(result)

["Brian_Jenn_Johnson_Bethel_-_After_All_These_Years_CeeNaija.com_.mp3", "Hillsong_Worship_-_This_is_The_Air_I_Breathe_CeeNaija.com_ - Copy.mp3", "Hillsong_Worship_-_This_is_The_Air_I_Breathe_CeeNaija.com_.mp3", "Hillsong_Worship_-_Unending_Love_CeeNaija.com_.mp3", "Jenn_Johnson_Bethel_-_Tremble_CeeNaija.com_.mp3", "Sub1"]
